# 01 · Diagnostic qualité des fichiers FAO

**Projet FAO DataLab — analyse mondiale de la sécurité alimentaire** · Patrick MWANIA MAZEMBE

Avant toute analyse, on mesure la qualité des 5 fichiers sources avec la classe `ProfileurFAO`
(héritée de `DataProfiler`, fichier `src/profiler.py`). Le diagnostic couvre les 4 familles d'anomalies :
valeurs manquantes, doublons et clés, incohérences métier, valeurs atypiques.
Chaque anomalie relevée ici reçoit une décision de traitement, appliquée dans le notebook 02.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from profiler import ProfileurFAO
from fao_pipeline import charger

pd.set_option("display.max_columns", 20)
sources = charger(RACINE / "data")
profils = {nom: ProfileurFAO(df, nom) for nom, df in sources.items()}
{nom: df.shape for nom, df in sources.items()}

{'vegetaux': (104871, 14), 'animaux': (37166, 14), 'cereales': (891, 14), 'population': (175, 14), 'sousalimentation': (1020, 15)}

## 1. Vue d'ensemble

Un rapport chiffré par fichier : taille, zones couvertes, périodes, valeurs manquantes ou non numériques,
doublons sur la clé métier (zone × produit × élément × année), valeurs négatives et unités multiples par élément.

In [2]:
rapport = pd.DataFrame([p.rapport() for p in profils.values()]).set_index("source")
rapport

,lignes,colonnes,cellules_nan,doublons_complets,zones,periodes,valeur_nan,valeur_texte,doublons_metier,valeurs_negatives,unites_multiples
source,,,,,,,,,,,
vegetaux,104871,14,0,0,175,2013,0,0,0,712,0
animaux,37166,14,0,0,175,2013,0,0,0,35,0
cereales,891,14,0,0,167,2013,0,0,0,0,0
population,175,14,174,0,175,2013,0,0,0,0,0
sousalimentation,1020,15,1435,0,204,"2012-2014, 2013-2015, 2014-2016, 2015-2017, 2016-2018",530,115,0,0,0


## 2. Valeurs manquantes et valeurs non numériques

Les bilans alimentaires (végétaux, animaux, céréales) et la population sont complets sur la colonne `Valeur`.
Toutes les valeurs manquantes sont dans le fichier de sous-alimentation. On regarde ce qu'elles recouvrent.

In [3]:
sa = sources["sousalimentation"]
print("Valeurs non numériques dans 'Valeur' :")
print(profils["sousalimentation"].valeurs_non_numeriques().to_string())
print()
print("Symboles FAO des lignes sans valeur :")
vides = sa[sa["Valeur"].isna()]
print(vides.groupby(["Symbole", "Description du Symbole"]).size().to_string())

Valeurs non numériques dans 'Valeur' :
Valeur
<0.1    115

Symboles FAO des lignes sans valeur :
Symbole  Description du Symbole
NR       Non rapportée             230
NV       Non disponible            185


**Lecture.** Deux situations très différentes se cachent derrière ces « trous » :

- **`<0.1` (115 lignes)** : la valeur existe, elle signifie « moins de 100 000 personnes sous-alimentées ».
  Un `pd.to_numeric(errors="coerce")` la transforme en vide : c'est l'erreur de la première version du projet.
- **Cellules vides** : symbole **NR** (« non rapportée ») ou **NV** (« non disponible »).

On vérifie ci-dessous à quels pays correspondent les NR : s'ils sont surtout riches, l'absence n'est pas aléatoire.

In [4]:
pop = sources["population"].set_index("Code zone")["Valeur"]
kcal = (sources["vegetaux"].query("Élément == 'Disponibilité alimentaire (Kcal/personne/jour)'")
        .groupby("Code zone")["Valeur"].sum()
        + sources["animaux"].query("Élément == 'Disponibilité alimentaire (Kcal/personne/jour)'")
        .groupby("Code zone")["Valeur"].sum())
p13 = sa[sa["Année"] == "2012-2014"].copy()
p13["Statut"] = p13["Symbole"].where(p13["Valeur"].isna(), "valeur")
p13.loc[p13["Valeur"] == "<0.1", "Statut"] = "<0.1"
p13["Kcal_total"] = p13["Code zone"].map(kcal)
p13.groupby("Statut").agg(zones=("Zone", "size"), kcal_moyenne=("Kcal_total", "mean")).round(0)

,zones,kcal_moyenne
Statut,,
<0.1,23,2 853.00
NR,47,3 353.00
NV,37,2 524.00
valeur,97,2 630.00


In [5]:
print("Zones NR (2012-2014) :")
print(", ".join(sorted(p13.loc[p13["Statut"] == "NR", "Zone"])))

Zones NR (2012-2014) :
Allemagne, Australie, Autriche, Azerbaïdjan, Belgique, Bosnie-Herzégovine, Brésil, Bélarus, Canada, Chine - RAS de Hong-Kong, Croatie, Cuba, Danemark, Espagne, Finlande, France, Fédération de Russie, Grèce, Hongrie, Irlande, Islande, Israël, Italie, Japon, Kazakhstan, Koweït, Lettonie, Lituanie, Luxembourg, Malte, Monténégro, Norvège, Nouvelle-Zélande, Pays-Bas, Pologne, Portugal, Roumanie, Royaume-Uni, République de Corée, Slovénie, Suisse, Suède, Tchéquie (la), Turquie, Ukraine, Uruguay, États-Unis d'Amérique


**Lecture.** Les zones « non rapportées » sont surtout des pays à haut revenu (Europe, Amérique du Nord, Japon…),
avec une disponibilité calorique moyenne nettement supérieure aux pays renseignés.
La FAO ne publie pas d'estimation quand la sous-alimentation y est très faible.
Ce ne sont donc **pas** des valeurs manquantes au hasard : les remplacer par une moyenne serait une erreur.

## 3. Doublons et clés de jointure

In [6]:
for nom, p in profils.items():
    print(f"{nom:17s} doublons complets = {p.doublons_complets():>3}   doublons clé métier = {p.doublons_metier():>3}")

vegetaux          doublons complets =   0   doublons clé métier =   0
animaux           doublons complets =   0   doublons clé métier =   0
cereales          doublons complets =   0   doublons clé métier =   0
population        doublons complets =   0   doublons clé métier =   0
sousalimentation  doublons complets =   0   doublons clé métier =   0


In [7]:
# Couverture géographique : zones présentes dans un fichier et pas dans l'autre
ecart = profils["population"].ecart_couverture(profils["sousalimentation"])
for k, v in ecart.items():
    print(f"{k} ({len(v)}) : {', '.join(v)}")
print()
ecart_veg = profils["vegetaux"].ecart_couverture(profils["population"])
print({k: len(v) for k, v in ecart_veg.items()})

seulement dans population (0) : 
seulement dans sousalimentation (29) : Andorre, Bahreïn, Bhoutan, Burundi, Comores, Groenland, Guinée équatoriale, Libye, Micronésie (États fédérés de), Nauru, Nioué, Palaos, Palestine, Papouasie-Nouvelle-Guinée, Porto Rico, Qatar, République arabe syrienne, République démocratique du Congo, Samoa américaines, Seychelles, Singapour, Somalie, Soudan du Sud, Tokélaou, Tonga, Tuvalu, Érythrée, Îles Cook, Îles Marshall

{'seulement dans vegetaux': 0, 'seulement dans population': 0}


**Lecture.** Végétaux, animaux et population couvrent exactement les mêmes 175 zones : c'est notre socle.
29 zones suivies pour la sous-alimentation n'ont pas de bilan alimentaire FAO pour 2013 (Libye, Syrie,
RD Congo, Somalie…). Ce n'est pas une erreur de jointure : on ne peut pas les analyser, mais on les **liste**.

In [8]:
# Agrégat « Chine » : est-il la somme de ses composantes ?
chine = sources["population"][sources["population"]["Zone"].str.startswith("Chine")][["Code zone", "Zone", "Valeur"]]
print(chine.to_string(index=False))
composantes = chine.loc[chine["Code zone"] != 351, "Valeur"].sum()
print(f"\nSomme des 4 composantes : {composantes:,.0f} milliers ; agrégat 'Chine' : {chine.loc[chine['Code zone'] == 351, 'Valeur'].iloc[0]:,.0f} milliers")

 Code zone                      Zone  Valeur
       351                     Chine 1416667
        96  Chine - RAS de Hong-Kong    7204
       128      Chine - RAS de Macao     566
        41       Chine, continentale 1385567
       214 Chine, Taiwan Province de   23330

Somme des 4 composantes : 1,416,667 milliers ; agrégat 'Chine' : 1,416,667 milliers


**Lecture.** La zone « Chine » (code 351) est l'agrégat de Chine continentale, Taïwan, Hong-Kong et Macao,
présents par ailleurs. La garder ajouterait 1,4 milliard de personnes en double à tout total mondial.

## 4. Incohérences métier : périodes, unités, valeurs négatives

In [9]:
for nom, p in profils.items():
    print(f"{nom:17s} périodes : {', '.join(p.periodes())}")

vegetaux          périodes : 2013
animaux           périodes : 2013
cereales          périodes : 2013
population        périodes : 2013
sousalimentation  périodes : 2012-2014, 2013-2015, 2014-2016, 2015-2017, 2016-2018


In [10]:
print("Valeurs négatives par élément :")
for nom in ["vegetaux", "animaux"]:
    print(f"- {nom} :", profils[nom].valeurs_negatives().to_dict())
print()
neg = sources["vegetaux"][(sources["vegetaux"]["Valeur"] < 0) & (sources["vegetaux"]["Élément"].str.contains("Kcal|protéines"))]
print("Disponibilités alimentaires négatives :")
print(neg[["Zone", "Produit", "Élément", "Valeur"]].to_string(index=False))
print()
print("Éléments avec plusieurs unités :",
      {nom: int((p.unites_par_element()["unites"].str.len() > 1).sum()) for nom, p in profils.items()})

Valeurs négatives par élément :
- vegetaux : {'Variation de stock': 599, 'Disponibilité intérieure': 105, 'Importations - Quantité': 1, 'Exportations - Quantité': 1, 'Nourriture': 1, 'Disponibilité alimentaire en quantité (kg/personne/an)': 1, 'Disponibilité alimentaire (Kcal/personne/jour)': 1, 'Disponibilité de protéines en quantité (g/personne/jour)': 1, 'Disponibilité de matière grasse en quantité (g/personne/jour)': 1, 'Traitement': 1}
- animaux : {'Variation de stock': 29, 'Disponibilité intérieure': 5, 'Traitement': 1}

Disponibilités alimentaires négatives :
 Zone Produit                                                  Élément  Valeur
Japon  Avoine           Disponibilité alimentaire (Kcal/personne/jour)  -21.00
Japon  Avoine Disponibilité de protéines en quantité (g/personne/jour)   -0.37

Éléments avec plusieurs unités : {'vegetaux': 0, 'animaux': 0, 'cereales': 0, 'population': 0, 'sousalimentation': 0}


**Lecture.**

- Les bilans alimentaires et la population sont **2013**, la sous-alimentation couvre cinq périodes de 3 ans
  (2012-2014 à 2016-2018). Pour comparer des choses comparables, on retiendra **2012-2014**, centrée sur 2013.
- La plupart des valeurs négatives sont des **« Variation de stock »** (628 lignes) : un pays qui puise dans
  ses stocks a une variation négative. C'est légitime, on n'y touche pas. La première version annonçait les
  « nettoyer » sur l'élément Production, où il n'y en a aucune.
- 110 « Disponibilité intérieure » négatives : solde comptable (production + importations − exportations ± stocks).
  Peu lisible, mais cette variable n'entre pas dans nos indicateurs : conservée.
- **Une vraie anomalie** : l'avoine au Japon a des importations, une nourriture et une disponibilité
  alimentaire négatives (−21 kcal/personne/jour). C'est impossible : exclue des sommes de kcal et de protéines.
- Une seule unité par élément : pas de conversion à prévoir.

## 5. Valeurs atypiques

On cherche les valeurs extrêmes de disponibilité calorique par produit (au-delà de Q3 + 3 × IQR).

In [11]:
extremes = profils["vegetaux"].valeurs_extremes("Disponibilité alimentaire (Kcal/personne/jour)")
print(f"{len(extremes)} valeurs extrêmes sur {len(sources['vegetaux'].query('Élément == "Disponibilité alimentaire (Kcal/personne/jour)"'))} lignes kcal végétales")
extremes.head(10)

998 valeurs extrêmes sur 10607 lignes kcal végétales


,Zone,Produit,Valeur_num,Unité
8512,Bangladesh,Riz (Eq Blanchi),1 711.00,Kcal/personne/jour
7361,Azerbaïdjan,Blé,1 666.00,Kcal/personne/jour
98521,Tunisie,Blé,1 576.00,Kcal/personne/jour
17425,Cambodge,Riz (Eq Blanchi),1 518.00,Kcal/personne/jour
99170,Turkménistan,Blé,1 469.00,Kcal/personne/jour
1709,Algérie,Blé,1 424.00,Kcal/personne/jour
82611,République démocratique populaire lao,Riz (Eq Blanchi),1 408.00,Kcal/personne/jour
63523,Maroc,Blé,1 392.00,Kcal/personne/jour
102484,Viet Nam,Riz (Eq Blanchi),1 390.00,Kcal/personne/jour
56164,Lesotho,Maïs,1 379.00,Kcal/personne/jour


**Lecture.** Les valeurs les plus hautes correspondent à des aliments de base (riz en Asie, blé au Maghreb,
maïs en Afrique australe) : ce sont des réalités alimentaires, pas des erreurs. On les conserve.

## 6. Synthèse : de l'anomalie à la décision

| # | Anomalie constatée | Ampleur | Décision (notebook 02) |
|---|---|---|---|
| 1 | `Valeur = "<0.1"` dans la sous-alimentation | 115 lignes (23 zones en 2012-2014) | Ne pas convertir en vide ni en 0 : garder une **borne haute** du taux (0,1 M / population) |
| 2 | Valeurs vides NR / NV | 47 NR + 8 NV zones en 2012-2014 | Garder le statut FAO ; imputation par modèle (notebook 04), signalée comme telle |
| 3 | Agrégat « Chine » (351) en double avec ses composantes | 1,4 milliard de personnes | Exclure l'agrégat |
| 4 | 29 zones sans bilan alimentaire | 29 zones | Hors périmètre, listées dans le journal |
| 5 | Périodes différentes entre fichiers | 2013 vs 2012-2014 … 2016-2018 | Retenir 2012-2014 (centrée sur 2013) |
| 6 | Valeurs négatives de Variation de stock / Disponibilité intérieure | 628 + 110 lignes | Légitimes : conservées |
| 6 bis | Disponibilité alimentaire négative (avoine, Japon) | 1 produit | Impossible : exclue des sommes de kcal et protéines |
| 7 | Valeurs extrêmes de kcal par produit | voir §5 | Réalités alimentaires : conservées |